# Titans Phase 1 — T4 synthetic screening
Select Tesla T4, enable Internet and grant GITHUB_TOKEN. Save & Run All publishes completed rows, including partial results on timeout.

In [ ]:
from pathlib import Path
import json
import shutil
import subprocess
import sys
import tempfile
import uuid

remote = 'https://github.com/ArcheLabs/mini-cells.git'
branch = 'research/minicells-08-runtime'
run_id = 'titans-phase1-screen-' + uuid.uuid4().hex[:12]
output = Path('/kaggle/working') / run_id
output.mkdir()
with tempfile.TemporaryDirectory() as tmp:
    repo = Path(tmp) / 'repo'
    subprocess.run(['git', 'clone', '--depth=1', '--branch', branch, remote, str(repo)],
                   check=True, timeout=300)
    source = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=repo, text=True).strip()
    sys.path.insert(0, str(repo))
    from scripts.research.publish_experiment_results import (
        ExperimentSpec, load_github_token, prepare_artifacts, push_results,
    )
    load_github_token('GITHUB_TOKEN')
    log = output / 'raw.log'
    try:
        with log.open('w') as stream:
            subprocess.run([sys.executable, str(repo / 'research/titans/phase1/screen.py'),
                            str(output), '--device', 'cuda'], cwd=repo, stdout=stream, stderr=subprocess.STDOUT,
                           check=True, timeout=9600)
        decision = {'status': 'SCREEN_EXECUTED'}
    except (subprocess.CalledProcessError, subprocess.TimeoutExpired) as exc:
        decision = {'status': 'SCREEN_INTERRUPTED', 'error': str(exc)}
    from collections import defaultdict
    import statistics
    grouped = defaultdict(list)
    if (output / 'episodes.jsonl').exists():
        for line in (output / 'episodes.jsonl').read_text().splitlines():
            try:
                row = json.loads(line)
            except json.JSONDecodeError:
                continue
            grouped[f"{row['n']}/{row['method']}"].append(row)
    summary = {}
    for key, rows in grouped.items():
        valid = [row for row in rows if 'recall' in row]
        summary[key] = {'episodes': len(rows), 'failures': len(rows) - len(valid),
                        'means': {metric: statistics.mean(row[metric] for row in valid)
                                  for metric in ('recall', 'mrr', 'margin', 'hard_fpr', 'update_seconds')
                                  if valid}}
    (output / 'summary.json').write_text(json.dumps(summary, indent=2))
    if decision['status'] == 'SCREEN_EXECUTED':
        decision['status'] = ('SCREEN_COMPLETE' if len(summary) == 20 and all(
            row['episodes'] == 60 and row['failures'] == 0 for row in summary.values())
            else 'SCREEN_INCOMPLETE')
    result_branch = 'kaggle/' + run_id
    decision.update(format='minicells.titans.phase1.screen.v1', source_commit=source, run_id=run_id)
    (output / 'decision.json').write_text(json.dumps(decision, indent=2))
    stage = repo / 'results/titans-phase1-screen'
    stage.mkdir(parents=True)
    names = ('decision.json', 'raw.log') + tuple(
        name for name in ('config.json', 'episodes.jsonl', 'summary.json') if (output / name).exists())
    for name in names:
        shutil.copy2(output / name, stage / name)
    spec = ExperimentSpec('results/titans-phase1-screen', 'artifacts/experiments/titans-phase1/' + run_id,
                          names, result_branch, decision['format'])
    dest = prepare_artifacts(repo, 'titans-phase1', spec, None)
    push_results(repo, dest, 'titans-phase1', result_branch, 'GITHUB_TOKEN', 'origin/main')
    receipt = {'status': 'REMOTE_SHA_VERIFIED', 'branch': result_branch,
               'commit': subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=repo, text=True).strip()}
    (output / 'publication.json').write_text(json.dumps(receipt, indent=2))
    print(decision, receipt, flush=True)
    if decision['status'] != 'SCREEN_COMPLETE':
        raise RuntimeError(f'Inspect {log}')
